# 01 – Đánh giá CV Parser (Module A)

P/R/F1 (micro) của từng tầng trích kỹ năng trên 50 CV gán nhãn tay, và MAE số năm kinh nghiệm.

Đầu vào: `backend/seed/cv_files/*` (PDF/DOCX/TXT) và `backend/seed/labels/labels_cv_skills.csv` (`cv_file,skills[,years_exp]`).

In [1]:
import os, sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "backend"))

# Có thể đổi bằng biến môi trường khi chạy thử trên dữ liệu khác
DATA = Path(os.getenv("TH3_DATA_DIR", ROOT / "backend/seed/data"))
LABELS = Path(os.getenv("TH3_LABELS_DIR", ROOT / "backend/seed/labels"))
OUTDIR = Path(os.getenv("TH3_OUT_DIR", ROOT / "docs/report/tables"))
MODELS_OUT = Path(os.getenv("TH3_MODELS_OUT", ROOT / "backend/models_store"))
OUTDIR.mkdir(parents=True, exist_ok=True)


In [2]:
import pandas as pd
from app.ai import datasets, evaluate
from app.ai.embedder import get_embedder
from app.ai.skills import SkillIndex

CV_DIR = Path(os.getenv("TH3_CV_DIR", ROOT / "backend/seed/cv_files"))
skills_csv, aliases_csv = DATA / "skills.csv", DATA / "skill_aliases.csv"
index = SkillIndex.from_csv(skills_csv, aliases_csv) if skills_csv.exists() else SkillIndex.default()
print("Số kỹ năng trong từ điển:", len(index.names), "| số cụm khoá:", len(index.phrases))
gold, years = datasets.load_cv_labels(LABELS / "labels_cv_skills.csv")
print("Số CV có nhãn:", len(gold))

Số kỹ năng trong từ điển: 81 | số cụm khoá: 200
Số CV có nhãn: 50


In [3]:
table = evaluate.evaluate_parser(CV_DIR, gold, years, index=index, embedder=get_embedder())
table.round(4)

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Could not reach the Hub (Cannot reach https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE` environment variable.). Using cached commit hash for 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:  48%|████▊     | 96/199 [00:00<00:00, 955.86it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1497.46it/s]

,tier,precision,recall,f1,n_cv
0,T1,0.9952,0.9523,0.9733,50
1,T1+2,0.9954,0.9795,0.9874,50
2,T1+2+3,0.9886,0.9886,0.9886,50


In [4]:
print("MAE số năm kinh nghiệm:", table.attrs.get("years_mae"))
table.to_csv(OUTDIR / "parser_eval.csv", index=False)
table.round(4)

MAE số năm kinh nghiệm: 0.11800000000000001


,tier,precision,recall,f1,n_cv
0,T1,0.9952,0.9523,0.9733,50
1,T1+2,0.9954,0.9795,0.9874,50
2,T1+2+3,0.9886,0.9886,0.9886,50
